# Does the asymmetry plateau?

One question, one run. The fabrication-to-other ratio falls monotonically with
detector capacity:

| detector | ratio | without H1 |
|---|---|---|
| tfidf | 3.1 | 2.7 |
| DeBERTa-v3, 512 | 2.4 | 2.2 |
| ModernBERT-base, 1,280 | 1.7 | 1.5 |
| ModernBERT-base, 2,048 | 1.4 | **1.1** |

A ratio of 1.0 means fabrication degrades no more than omission or distortion,
i.e. no asymmetry at all. At 1.1 the strongest configuration is already close to
it, so a reviewer can fairly ask whether one more step crosses the line.

**This is optional, and deliberately so.** The paper states that it cannot tell
whether the ratio plateaus above one or reaches it. An acknowledged open
question is defensible; a claim disproved in review is not. If you run this and
the ratio lands at or near 1.0, the headline finding needs reframing, so do not
start it close to a deadline.

**Runtime:** ModernBERT-large, four conditions (random plus three LOSO folds),
about 4 h on an A100. Held-out-strategy adds another hour and is not needed for
the ratio.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CANDIDATES = [
    '/content/drive/Shareddrives/RESEARCH/2026/PROMPTS/RESULTS',
    '/content/drive/Shareddrives/RESEARCH/Research/2026/PROMPTS/RESULTS',
]
ok = lambda p: os.path.isdir(os.path.join(p, 'EVALUATION-RESULT'))
RESULTS_ROOT = next((p for p in CANDIDATES if ok(p)), None)
if RESULTS_ROOT is None:
    raise SystemExit('Set RESULTS_ROOT by hand.')

OUT = f'{RESULTS_ROOT}/ANALYSIS-OUTPUT-RESULT/iclr_benchmark'
os.environ['HALLUBENCH_OUT'] = OUT
LABELS = f'{OUT}/benchmark_labels.csv.gz'
print('OUT    =', OUT)
print('labels =', 'found' if os.path.exists(LABELS) else 'MISSING')

In [ ]:
!pip -q install "transformers>=4.48" accelerate
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE — Runtime > Change runtime type > GPU')

In [ ]:
%%writefile /content/06_encoder_detector.py
#!/usr/bin/env python3
"""
Step 6 - fine-tuned encoder detector. RUN THIS ON A GPU (Colab A100 is enough).

Addresses the strongest objection to the paper: that the transfer collapse is
an artifact of linear detectors rather than a property of the task. Fine-tunes
one encoder per split with six per-type heads plus an ANY head, on
[report] [SEP] [reference], and evaluates on the standard, leave-one-source-out
(all three folds), and held-out-strategy splits.

Requires network access to download model weights, so it cannot run in the
offline analysis container.

    pip install "transformers>=4.48" torch scikit-learn accelerate
    python3 06_encoder_detector.py --model answerdotai/ModernBERT-base

CONTEXT LENGTH MATTERS HERE. Reports average ~1,500 tokens. The judge that
produced the labels saw 3,000 characters of report (~750 tokens) plus 1,500 of
reference (~375), so --max_len 1280 gives the detector exactly the judge's
view. At 512 the detector sees less than the judge did and the comparison is
unfair to it. ModernBERT handles 8,192 tokens, so 1,280 costs nothing.

Writes: encoder_results.csv  (same schema as baseline_results.csv, so the
        existing report and figure code consumes it unchanged)

Runtime guide, A100, ModernBERT-base, max_len 1280, bs 8, 2 epochs:
    standard split          ~60 min
    3 LOSO folds            ~150 min
    held-out strategy       ~60 min
Roughly 4.5 h in total. For the validation pass use
    --model roberta-base --max_len 512 --bs 16 --epochs 1
which takes about 20 minutes and only checks that the loop runs.
"""
import argparse, os
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

ap = argparse.ArgumentParser()
ap.add_argument('--model', default='answerdotai/ModernBERT-base')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/encoder_results.csv')
ap.add_argument('--max_len', type=int, default=1280,
                help='1280 matches what the judge saw; raise it to test whether '
                     'the detector benefits from more than the judge had')
ap.add_argument('--epochs', type=int, default=2)
ap.add_argument('--bs', type=int, default=8,
                help='lower than usual because of the long context')
ap.add_argument('--lr', type=float, default=2e-5)
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

torch.manual_seed(args.seed); np.random.seed(args.seed)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
if dev == 'cpu':
    print('WARNING: no GPU visible. This will take many hours.')

df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')
tok = AutoTokenizer.from_pretrained(args.model)
_cap = getattr(tok, 'model_max_length', 512)
if _cap and _cap < args.max_len and _cap < 100000:
    print(f'WARNING: {args.model} caps at {_cap} tokens but --max_len is '
          f'{args.max_len}. Reports will be cut below what the judge saw. '
          f'Use a long-context model (ModernBERT, Longformer) or lower '
          f'--max_len and say so in the paper.')


class Reports(Dataset):
    """Report and its reference annotation as a sentence pair."""
    def __init__(self, frame):
        self.a = frame.model_output.tolist()
        self.b = frame.ground_truth.tolist()
        self.y = frame[TARGETS].to_numpy(dtype='float32')

    def __len__(self):
        return len(self.a)

    def __getitem__(self, i):
        enc = tok(self.a[i], self.b[i], truncation=True, max_length=args.max_len,
                  padding='max_length', return_tensors='pt')
        return ({k: v.squeeze(0) for k, v in enc.items()},
                torch.tensor(self.y[i]))


class MultiHead(torch.nn.Module):
    """One shared encoder, seven independent binary heads."""
    def __init__(self, name, n=len(TARGETS)):
        super().__init__()
        # force fp32: some checkpoints (DeBERTa-v3) declare a fp16 dtype in
        # their config, which makes GradScaler refuse to unscale gradients
        self.enc = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        d = self.enc.config.hidden_size
        self.drop = torch.nn.Dropout(0.1)
        self.heads = torch.nn.Linear(d, n)

    def forward(self, **kw):
        h = self.enc(**kw).last_hidden_state[:, 0]     # [CLS]
        return self.heads(self.drop(h))


def run(train_mask, test_mask, tag):
    tr, te = df[train_mask].reset_index(drop=True), df[test_mask].reset_index(drop=True)
    print(f'\n== {tag}: train {len(tr):,}  test {len(te):,}', flush=True)

    model = MultiHead(args.model).to(dev)
    dl_tr = DataLoader(Reports(tr), batch_size=args.bs, shuffle=True, num_workers=2)
    dl_te = DataLoader(Reports(te), batch_size=args.bs * 2, num_workers=2)

    # class weights per head, matching the balanced logistic baselines
    pos = tr[TARGETS].mean().to_numpy()
    w = torch.tensor(((1 - pos) / np.clip(pos, 1e-6, None)).astype('float32')).to(dev)
    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=w)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr)
    steps = len(dl_tr) * args.epochs
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)

    # bf16 where the GPU supports it (A100 and newer): same dynamic range as
    # fp32, so no loss scaling is needed and GradScaler is skipped entirely.
    use_bf16 = dev == 'cuda' and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda' and not use_bf16))
    print(f'   precision: {"bf16" if use_bf16 else ("fp16+scaler" if dev=="cuda" else "fp32")}',
          flush=True)

    model.train()
    for ep in range(args.epochs):
        for i, (x, y) in enumerate(dl_tr):
            x = {k: v.to(dev) for k, v in x.items()}; y = y.to(dev)
            opt.zero_grad()
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=(dev == 'cuda')):
                loss = lossf(model(**x), y)
            if scaler.is_enabled():
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            sch.step()
            if i % 100 == 0:
                print(f'   ep{ep} step {i}/{len(dl_tr)} loss {loss.item():.4f}', flush=True)

    model.eval(); P = []
    with torch.no_grad():
        for x, _ in dl_te:
            x = {k: v.to(dev) for k, v in x.items()}
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=(dev == 'cuda')):
                P.append(torch.sigmoid(model(**x)).float().cpu().numpy())
    P = np.vstack(P)

    rows = []
    for j, t in enumerate(TARGETS):
        y = te[t].to_numpy()
        if len(np.unique(y)) < 2:
            continue
        rows.append({'features': 'encoder', 'split': tag, 'target': t,
                     'auc': roc_auc_score(y, P[:, j]),
                     'f1': f1_score(y, (P[:, j] >= 0.5).astype(int)),
                     'pos_rate_test': float(y.mean())})
    del model; torch.cuda.empty_cache()
    return rows


out = []
out += run(df.split_random == 'train', df.split_random == 'test', 'random')
for held in ['Claude', 'GPT', 'Gemini']:
    out += run(df.model != held, df.model == held, f'heldout_model_{held}')
out += run(df.split_heldout_technique == 'train',
           df.split_heldout_technique == 'test', 'heldout_technique')

res = pd.DataFrame(out)
res.to_csv(args.out, index=False)
print('\n' + res.pivot_table(index='split', columns='target',
                             values='auc')[TARGETS].round(3).to_markdown())
print('\nwrote ->', args.out)


## Run

`--bs 4` because large needs roughly double the memory of base. If it runs out,
drop to 2; the result is unaffected, only the wall clock.

In [ ]:
!python3 /content/06_encoder_detector.py \
    --model answerdotai/ModernBERT-large --max_len 1280 --bs 4 --epochs 2 \
    --labels "$LABELS" --out "$OUT/encoder_results_modernbert_large.csv" 

## The answer

Computes the ratio with and without H1 and places it in the series. The
`without H1` column is the one that matters: H1 is excluded from the headline
macro for low reliability, so the conservative reading of the trend uses H5 and
H6 only.

In [ ]:
import pandas as pd, numpy as np, os

H = ['H1','H2','H3','H4','H5','H6']
FAB, FAB_NO_H1 = ['H1','H5','H6'], ['H5','H6']
OMI, DIST = ['H3'], ['H2','H4']

def ratios(path, label):
    """Fabrication drop over the larger of the omission and distortion drops."""
    d = pd.read_csv(path)
    r = d[d.split == 'random'].set_index('target').auc
    l = d[d.split.str.startswith('heldout_model_')].groupby('target').auc.mean()
    drop = {h: l[h] - r[h] for h in H}
    fab   = np.mean([drop[h] for h in FAB])
    fab2  = np.mean([drop[h] for h in FAB_NO_H1])
    other = min(np.mean([drop[h] for h in OMI]),
                np.mean([drop[h] for h in DIST]))   # smaller magnitude drop
    other = max(np.mean([drop[h] for h in OMI]),
                np.mean([drop[h] for h in DIST]))   # conservative denominator
    return {'detector': label, 'fab drop': round(fab, 3),
            'ratio': round(fab / other, 2), 'ratio no H1': round(fab2 / other, 2),
            'ANY (LOSO)': round(l['any_hallucination'], 3)}

rows = [
    {'detector':'tfidf (linear)','fab drop':-0.191,'ratio':3.1,'ratio no H1':2.7,'ANY (LOSO)':0.566},
    {'detector':'DeBERTa-v3, 512','fab drop':-0.104,'ratio':2.4,'ratio no H1':2.2,'ANY (LOSO)':0.729},
    {'detector':'ModernBERT-base, 1280','fab drop':-0.163,'ratio':1.7,'ratio no H1':1.5,'ANY (LOSO)':0.644},
    {'detector':'ModernBERT-base, 2048','fab drop':-0.145,'ratio':1.4,'ratio no H1':1.1,'ANY (LOSO)':0.642},
]
path = f'{OUT}/encoder_results_modernbert_large.csv'
if os.path.exists(path):
    rows.append(ratios(path, 'ModernBERT-large, 1280'))
else:
    print('large run not found — showing the existing series only')

t = pd.DataFrame(rows).set_index('detector')
t.to_csv(f'{OUT}/ratio_series.csv')
print(t.to_markdown())

if len(rows) == 5:
    prev, new = rows[-2]['ratio no H1'], rows[-1]['ratio no H1']
    print(f"\nratio without H1: {prev} -> {new}")
    if new <= 1.05:
        print("REACHES 1.0. The asymmetry does not survive at scale. The headline")
        print("claim needs reframing: report it as a property of the detectors")
        print("tested, not of hallucination type. Do not submit unchanged.")
    elif new >= prev - 0.05:
        print("PLATEAUS. The asymmetry is real and persists with capacity.")
        print("Replace the open question in Section 4.3 with this result.")
    else:
        print("STILL FALLING but above one. Report the series and keep the")
        print("open question; the trend is now evidence rather than speculation.")

## If you run it

Send me `ratio_series.csv` and the printed verdict. Section 4.3 currently says
we cannot tell whether the ratio plateaus above one or reaches it; whichever way
it lands, that sentence gets replaced with the answer.